In [4]:
import json
import joblib
import numpy as np
from konlpy.tag import Okt
from gensim.models import Word2Vec
from soynlp.tokenizer import LTokenizer

# 경로 설정 (필요에 따라 수정)
DB_PATH = "C:/kosmo/data/"
PATH = "C:/kosmo/test_data/IH/"
SCORE_PATH = "C:/kosmo/model/"
file_name = "movies_DB_partial.json"

# 1) 영화 데이터 불러오기
with open(DB_PATH + file_name, encoding="utf-8") as f:
    movies = json.load(f)

# 2) soynlp word_scores 불러오기 (Scores 객체 → float 딕셔너리로 변환 필요)
raw_word_scores = joblib.load(SCORE_PATH + "soynlp_word_scores.pkl")  # Scores 객체 딕셔너리 로드

# float 점수만 추출 (cohesion_forward 예시, 필요 시 다른 점수로 변경)
word_scores = {word: score.cohesion_forward for word, score in raw_word_scores.items()}

# 3) LTokenizer 생성
tokenizer = LTokenizer(scores=word_scores)

# 4) Okt 형태소 분석기 생성
okt = Okt()

# 5) 토큰화 함수 정의
def get_tokens(title, overview, genres, genre_weight=3):
    title = title or ""
    overview = overview or ""
    genres = genres or []

    tokens = []

    # 제목 토큰화: soynlp LTokenizer 사용
    title_tokens = tokenizer.tokenize(title)
    tokens.extend(title_tokens)

    # 줄거리 토큰화: konlpy Okt 사용 (명사, 동사, 형용사만)
    for word, pos in okt.pos(overview, stem=True, norm=True):
        if pos in ['Noun', 'Verb', 'Adjective']:
            tokens.append(word)

    # 장르 토큰 가중치 부여
    tokens.extend(genres * genre_weight)

    return tokens

# 6) 전체 영화 토큰화
keyword = []
for movie in movies:
    tokens = get_tokens(
        movie.get('title_ko'),
        movie.get('overview'),
        movie.get('genres'),
        genre_weight=3
    )
    keyword.append(tokens)

# 7) Word2Vec 모델 학습 (compute_loss=True 옵션 추가)
model = Word2Vec(
    keyword,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0,  # CBOW
    compute_loss=True  # 손실 계산 활성화
)

epochs = 10  # 원하는 에포크 수

previous_loss = 0
for epoch in range(epochs):
    model.train(keyword, total_examples=len(keyword), epochs=1)
    current_loss = model.get_latest_training_loss()
    # 에포크별 손실은 현재 누적 손실에서 이전 누적 손실 차이
    epoch_loss = current_loss - previous_loss
    previous_loss = current_loss
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}")

# 8) 검색용 제목 토큰 저장 (줄거리/장르 제외)
title_tokens = [tokenizer.tokenize(movie.get('title_ko', '')) for movie in movies]
joblib.dump(title_tokens, PATH + "title_tokens.pkl")

# 9) 영화 벡터 생성 함수 정의
def get_movie_vector(tokens, model):
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    return np.mean(np.array(vectors), axis=0) if vectors else np.zeros(model.vector_size)

# 10) 영화 벡터 계산
movie_vectors = [get_movie_vector(tokens, model) for tokens in keyword]

# 11) 모델 및 데이터 저장
joblib.dump(model, PATH + "word2vec_movie.model")
joblib.dump(movie_vectors, PATH + "movie_vectors.pkl")
joblib.dump(movies, PATH + "movies.pkl")
joblib.dump(keyword, PATH + "keyword.pkl")

print("✅ soynlp 기반 Word2Vec 학습 및 전체 데이터 저장 완료")


Epoch 1/10, Loss: 0.0000
Epoch 2/10, Loss: 0.0000
Epoch 3/10, Loss: 0.0000
Epoch 4/10, Loss: 0.0000
Epoch 5/10, Loss: 0.0000
Epoch 6/10, Loss: 0.0000
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0000
Epoch 10/10, Loss: 0.0000
✅ soynlp 기반 Word2Vec 학습 및 전체 데이터 저장 완료


In [ ]:
import joblib

# 1) 모델 불러오기 (경로 수정하세요)
model_path = "C:/kosmo/test_data/IH/word2vec_movie.model"
model = joblib.load(model_path)

# 2) 단어 유사도 출력 함수
def print_similarity(model, word1, word2):
    try:
        score = model.wv.similarity(word1, word2)
        print(f"'{word1}' 와(과) '{word2}' 의 유사도: {score:.4f}")
    except KeyError as e:
        print(f"❌ 단어가 vocabulary에 없습니다: {e}")

# 3) 유사 단어 Top-N 출력 함수
def print_most_similar(model, word, topn=10):
    try:
        print(f"\n'{word}' 와(과) 유사한 단어 Top {topn}:")
        similar_words = model.wv.most_similar(word, topn=topn)
        for w, s in similar_words:
            print(f"  {w}: {s:.4f}")
    except KeyError as e:
        print(f"❌ 단어가 vocabulary에 없습니다: {e}")

# 4) 테스트 실행 예시
if __name__ == "__main__":
    print_similarity(model, "마블", "어벤져스")
    print_most_similar(model, "아이언맨", topn=5)

'마블' 와(과) '어벤져스' 의 유사도: 0.6264

아이언맨' 와(과) 유사한 단어 Top 5:
  헐크: 0.7128
  그린랜턴: 0.6691
  나노: 0.6653
  사이보그: 0.6634
  블랙위도우: 0.6632
